# Robustness — observation-noise sweep

Public reproducibility version for the manuscript *Mapping-Class-Group Learning for Diffeomorphisms*.

**Important:** the revised manuscript reports candidate selection by unpenalized validation loss (equivalently, `lambda = 0`). The original exploratory notebooks also logged a reference word-length penalty of `1e-7`; that legacy setting did not change the selected candidate in the reported runs.

Run Jupyter from the repository root so that relative output paths under `results/` resolve as intended.



# Step 6: Observation-noise sweep

This experiment examines the robustness of mapping-class selection to noisy observations.

The underlying target remains
\[
f=F_0\circ AB,
\]
with the baseline residual amplitude \(\varepsilon=0.05\).

Instead of observing the exact output \(f(X)\), we use wrapped Gaussian observation noise:
\[
Y^{\rm obs}
=
\left(f(X)+\sigma_{\rm noise}\xi\right)\bmod 1,
\qquad
\xi\sim N(0,I_2).
\]

We sweep
\[
\sigma_{\rm noise}
\in
\{0,\;0.01,\;0.02,\;0.05,\;0.10,\;0.20\}.
\]

All other settings are fixed:
- hidden mapping-class element: \(G_\star=AB\);
- residual amplitude: \(\varepsilon=0.05\);
- original 9-candidate set;
- uniform sampling on \(T^2\);
- \(2000/1000/1000\) train/validation/test samples;
- PNN architecture \(4\to128\to128\to2\);
- \(\tanh\) activations;
- residual scale \(0.1\);
- Adam, learning rate \(10^{-3}\);
- full-batch training;
- 3000 epochs per candidate;
- no early stopping;
- \(\lambda=10^{-7}\);
- five seeds \(0,1,2,3,4\).

For each seed, the same input samples and the same underlying Gaussian noise draws are reused across all noise levels, with only their scale changed. This paired design isolates the effect of \(\sigma_{\rm noise}\).

Candidate selection is based on the **noisy validation loss**. We additionally report:
1. correct-selection frequency;
2. noisy validation margin to the best wrong candidate;
3. clean test loss against the underlying noiseless target \(f(X)\);
4. noisy test loss against the observed noisy outputs;
5. mean and sample standard deviation across five seeds.


In [ ]:

import os
import time
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

QUICK_TEST = False

# Repository-relative output directory.
os.makedirs("results/observation_noise", exist_ok=True)

SEEDS = [0, 1, 2, 3, 4]
NOISE_LEVELS = [0.0, 0.01, 0.02, 0.05, 0.10, 0.20]

N_TRAIN = 2000
N_VAL   = 1000
N_TEST  = 1000

EPSILON = 0.05

EPOCHS = 3000
LR = 1e-3
HIDDEN_WIDTH = 128
RESIDUAL_SCALE = 0.1
# Revised manuscript: primary selector is unpenalized validation loss.
# The original 1e-7 value is retained only in archived/raw results.
LAMBDA_WORD = 0.0
if QUICK_TEST:
    SEEDS = [0]
    NOISE_LEVELS = [0.0, 0.05, 0.20]
    N_TRAIN = 200
    N_VAL = 100
    N_TEST = 100
    EPOCHS = 20

device = torch.device("cpu")

try:
    torch.use_deterministic_algorithms(True)
except Exception as e:
    print("Could not enforce deterministic algorithms:", e)

print("device:", device)
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("NOISE_LEVELS =", NOISE_LEVELS)
print("QUICK_TEST:", QUICK_TEST)


In [ ]:

# ------------------------------------------------------------
# Torus loss
# ------------------------------------------------------------

def torus_diff(pred, target):
    return (pred - target + 0.5) % 1.0 - 0.5

def torus_mse(pred, target):
    return torch.mean(torus_diff(pred, target) ** 2)


In [ ]:

# ------------------------------------------------------------
# Original 9 candidate mapping-class elements
# ------------------------------------------------------------

A = np.array([[1, 1],
              [0, 1]], dtype=np.int64)

A_INV = np.array([[ 1, -1],
                  [ 0,  1]], dtype=np.int64)

B = np.array([[1, 0],
              [1, 1]], dtype=np.int64)

B_INV = np.array([[ 1,  0],
                  [-1,  1]], dtype=np.int64)

TOKEN_MATRIX = {
    "A": A,
    "Ainv": A_INV,
    "B": B,
    "Binv": B_INV,
}

def word_matrix(tokens):
    M = np.eye(2, dtype=np.int64)
    for token in tokens:
        M = M @ TOKEN_MATRIX[token]
    return M

def matrix_map(M):
    Mt = torch.tensor(M.T, dtype=torch.float32, device=device)

    def G(x):
        return (x @ Mt) % 1.0

    return G

candidate_tokens = {
    "id":   [],
    "A":    ["A"],
    "A^-1": ["Ainv"],
    "B":    ["B"],
    "B^-1": ["Binv"],
    "AB":   ["A", "B"],
    "BA":   ["B", "A"],
    "A^2":  ["A", "A"],
    "B^2":  ["B", "B"],
}

candidates = {}

for name, tokens in candidate_tokens.items():
    M = word_matrix(tokens)
    candidates[name] = {
        "tokens": tokens,
        "matrix": M,
        "length": len(tokens),
        "map": matrix_map(M),
    }

# Verify distinct elements.
seen = {}
for name, info in candidates.items():
    key = tuple(info["matrix"].reshape(-1).tolist())
    if key in seen:
        raise ValueError(f"Duplicate element: {seen[key]} and {name}")
    seen[key] = name

display(
    pd.DataFrame(
        [
            {
                "candidate": name,
                "length": info["length"],
                "matrix": str(info["matrix"].tolist()),
            }
            for name, info in candidates.items()
        ]
    )
)

HIDDEN_WORD = "AB"


In [ ]:

# ------------------------------------------------------------
# Baseline residual deformation and clean target
# ------------------------------------------------------------

def F0(x, eps=0.05):
    u = (
        x[..., 0]
        + eps * torch.sin(2 * torch.pi * (x[..., 0] - x[..., 1]))
    ) % 1.0

    v = (
        x[..., 1]
        + eps * torch.sin(2 * torch.pi * x[..., 1])
    ) % 1.0

    return torch.stack([u, v], dim=-1)

def clean_target(x):
    return F0(
        candidates[HIDDEN_WORD]["map"](x),
        eps=EPSILON
    )

def add_wrapped_noise(y_clean, z, sigma):
    return (y_clean + sigma * z) % 1.0


In [ ]:

# ------------------------------------------------------------
# Periodic neural network
# ------------------------------------------------------------

class PeriodicResidualNN(torch.nn.Module):
    def __init__(self, hidden=128, residual_scale=0.1):
        super().__init__()
        self.residual_scale = residual_scale

        self.net = torch.nn.Sequential(
            torch.nn.Linear(4, hidden),
            torch.nn.Tanh(),
            torch.nn.Linear(hidden, hidden),
            torch.nn.Tanh(),
            torch.nn.Linear(hidden, 2),
        )

    def periodic_features(self, x):
        return torch.cat(
            [
                torch.cos(2 * torch.pi * x),
                torch.sin(2 * torch.pi * x),
            ],
            dim=-1,
        )

    def forward(self, x):
        features = self.periodic_features(x)
        dx = self.residual_scale * self.net(features)
        return (x + dx) % 1.0


In [ ]:

# ------------------------------------------------------------
# Candidate-wise training
# ------------------------------------------------------------

def train_one_candidate(
    candidate_name,
    x_train,
    y_train_obs,
    x_val,
    y_val_obs,
    x_test,
    y_test_clean,
    y_test_obs,
    seed,
):
    # Same initialization across candidates within one seed/noise level.
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = PeriodicResidualNN(
        hidden=HIDDEN_WIDTH,
        residual_scale=RESIDUAL_SCALE,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    G = candidates[candidate_name]["map"]

    with torch.no_grad():
        gx_train = G(x_train)
        gx_val = G(x_val)
        gx_test = G(x_test)

    model.train()

    for _ in range(EPOCHS):
        optimizer.zero_grad()

        pred = model(gx_train)
        loss = torus_mse(pred, y_train_obs)

        loss.backward()
        optimizer.step()

    model.eval()

    with torch.no_grad():
        pred_train = model(gx_train)
        pred_val = model(gx_val)
        pred_test = model(gx_test)

        train_loss_noisy = torus_mse(
            pred_train, y_train_obs
        ).item()

        val_loss_noisy = torus_mse(
            pred_val, y_val_obs
        ).item()

        test_loss_clean = torus_mse(
            pred_test, y_test_clean
        ).item()

        test_loss_noisy = torus_mse(
            pred_test, y_test_obs
        ).item()

    word_length = candidates[candidate_name]["length"]

    penalized_score = (
        val_loss_noisy
        + LAMBDA_WORD * word_length
    )

    return {
        "train_loss_noisy": train_loss_noisy,
        "val_loss_noisy": val_loss_noisy,
        "test_loss_clean": test_loss_clean,
        "test_loss_noisy": test_loss_noisy,
        "word_length": word_length,
        "penalized_score": penalized_score,
    }


In [ ]:

# ------------------------------------------------------------
# Run noise sweep
# ------------------------------------------------------------

all_rows = []
run_summaries = []

total_trainings = (
    len(SEEDS)
    * len(NOISE_LEVELS)
    * len(candidates)
)

completed_trainings = 0
global_start = time.time()

for seed in SEEDS:

    # --------------------------------------------------------
    # Paired input samples and paired Gaussian noise draws.
    # --------------------------------------------------------

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    x_train = torch.rand(N_TRAIN, 2, device=device)
    x_val   = torch.rand(N_VAL, 2, device=device)
    x_test  = torch.rand(N_TEST, 2, device=device)

    with torch.no_grad():
        y_train_clean = clean_target(x_train)
        y_val_clean   = clean_target(x_val)
        y_test_clean  = clean_target(x_test)

    # Draw base noise only once per seed.
    # The same z values are multiplied by each sigma.
    z_train = torch.randn(N_TRAIN, 2, device=device)
    z_val   = torch.randn(N_VAL, 2, device=device)
    z_test  = torch.randn(N_TEST, 2, device=device)

    for sigma in NOISE_LEVELS:

        with torch.no_grad():
            y_train_obs = add_wrapped_noise(
                y_train_clean, z_train, sigma
            )

            y_val_obs = add_wrapped_noise(
                y_val_clean, z_val, sigma
            )

            y_test_obs = add_wrapped_noise(
                y_test_clean, z_test, sigma
            )

        print("=" * 78)
        print(
            f"seed={seed} | sigma_noise={sigma:.3f} "
            f"| hidden={HIDDEN_WORD}"
        )
        print("=" * 78)

        rows_this_run = []

        for candidate_name in candidates:

            t0 = time.time()

            result = train_one_candidate(
                candidate_name,
                x_train,
                y_train_obs,
                x_val,
                y_val_obs,
                x_test,
                y_test_clean,
                y_test_obs,
                seed=seed,
            )

            row = {
                "seed": seed,
                "sigma_noise": sigma,
                "candidate": candidate_name,
                **result,
            }

            all_rows.append(row)
            rows_this_run.append(row)

            completed_trainings += 1

            elapsed = time.time() - global_start
            avg_per_training = elapsed / completed_trainings
            remaining = total_trainings - completed_trainings
            eta_min = avg_per_training * remaining / 60.0

            print(
                f"{candidate_name:6s} "
                f"val_noisy={result['val_loss_noisy']:.4e} "
                f"score={result['penalized_score']:.4e} "
                f"clean_test={result['test_loss_clean']:.4e} "
                f"({time.time()-t0:.1f}s; ETA ~{eta_min:.1f} min)"
            )

        run_df = pd.DataFrame(rows_this_run)

        target_row = run_df[
            run_df["candidate"] == HIDDEN_WORD
        ].iloc[0]

        wrong_df = run_df[
            run_df["candidate"] != HIDDEN_WORD
        ]

        best_wrong_idx = wrong_df[
            "val_loss_noisy"
        ].idxmin()

        best_wrong_row = wrong_df.loc[
            best_wrong_idx
        ]

        validation_margin = (
            best_wrong_row["val_loss_noisy"]
            - target_row["val_loss_noisy"]
        )

        selected_penalized = run_df.loc[
            run_df["penalized_score"].idxmin(),
            "candidate"
        ]

        selected_unpenalized = run_df.loc[
            run_df["val_loss_noisy"].idxmin(),
            "candidate"
        ]

        selected_row = run_df[
            run_df["candidate"] == selected_penalized
        ].iloc[0]

        run_summary = {
            "seed": seed,
            "sigma_noise": sigma,
            "selected_penalized": selected_penalized,
            "selected_unpenalized": selected_unpenalized,
            "correct_penalized": (
                selected_penalized == HIDDEN_WORD
            ),
            "correct_unpenalized": (
                selected_unpenalized == HIDDEN_WORD
            ),
            "target_val_loss_noisy": (
                target_row["val_loss_noisy"]
            ),
            "best_wrong_candidate": (
                best_wrong_row["candidate"]
            ),
            "best_wrong_val_loss_noisy": (
                best_wrong_row["val_loss_noisy"]
            ),
            "validation_margin": validation_margin,
            "selected_clean_test_loss": (
                selected_row["test_loss_clean"]
            ),
            "selected_noisy_test_loss": (
                selected_row["test_loss_noisy"]
            ),
        }

        run_summaries.append(run_summary)

        print("\nRun summary:")
        print(run_summary)

        # Autosave after every seed/noise condition.
        pd.DataFrame(all_rows).to_csv(
            "results/observation_noise/noise_sweep_all_candidate_results.csv",
            index=False,
        )

        pd.DataFrame(run_summaries).to_csv(
            "results/observation_noise/noise_sweep_run_summaries.csv",
            index=False,
        )

print("\nFull noise sweep completed.")


In [ ]:

# ------------------------------------------------------------
# Per-run results
# ------------------------------------------------------------

run_summary_df = pd.DataFrame(run_summaries)

display(
    run_summary_df.sort_values(
        ["sigma_noise", "seed"]
    )
)

print("\nSelection counts (lambda = 0):")
display(
    pd.crosstab(
        run_summary_df["sigma_noise"],
        run_summary_df["selected_penalized"],
    )
)


In [ ]:

# ------------------------------------------------------------
# Aggregate over five seeds
# ------------------------------------------------------------

def sample_sd(x):
    x = np.asarray(x, dtype=float)

    return (
        np.std(x, ddof=1)
        if len(x) > 1
        else np.nan
    )

summary_rows = []

for sigma in NOISE_LEVELS:

    d = run_summary_df[
        run_summary_df["sigma_noise"] == sigma
    ]

    summary_rows.append({
        "sigma_noise": sigma,
        "correct_penalized_count": int(
            d["correct_penalized"].sum()
        ),
        "n_seeds": len(d),
        "selection_rate": float(
            d["correct_penalized"].mean()
        ),
        "target_val_noisy_mean": float(
            d["target_val_loss_noisy"].mean()
        ),
        "target_val_noisy_sd": sample_sd(
            d["target_val_loss_noisy"]
        ),
        "margin_mean": float(
            d["validation_margin"].mean()
        ),
        "margin_sd": sample_sd(
            d["validation_margin"]
        ),
        "clean_test_mean": float(
            d["selected_clean_test_loss"].mean()
        ),
        "clean_test_sd": sample_sd(
            d["selected_clean_test_loss"]
        ),
        "noisy_test_mean": float(
            d["selected_noisy_test_loss"].mean()
        ),
        "noisy_test_sd": sample_sd(
            d["selected_noisy_test_loss"]
        ),
    })

summary_df = pd.DataFrame(summary_rows)

display(summary_df)

summary_df.to_csv(
    "results/observation_noise/noise_sweep_aggregate_summary.csv",
    index=False,
)


In [ ]:

# ------------------------------------------------------------
# Manuscript-friendly table
# ------------------------------------------------------------

paper_rows = []

for _, r in summary_df.iterrows():

    paper_rows.append({
        "sigma_noise": f"{r['sigma_noise']:.3f}",
        "Selected correctly": (
            f"{int(r['correct_penalized_count'])}/"
            f"{int(r['n_seeds'])}"
        ),
        "Noisy val. loss (mean ± SD)": (
            f"{r['target_val_noisy_mean']:.3e} ± "
            f"{r['target_val_noisy_sd']:.3e}"
        ),
        "Margin (mean ± SD)": (
            f"{r['margin_mean']:.3e} ± "
            f"{r['margin_sd']:.3e}"
        ),
        "Clean test loss (mean ± SD)": (
            f"{r['clean_test_mean']:.3e} ± "
            f"{r['clean_test_sd']:.3e}"
        ),
        "Noisy test loss (mean ± SD)": (
            f"{r['noisy_test_mean']:.3e} ± "
            f"{r['noisy_test_sd']:.3e}"
        ),
    })

paper_table_df = pd.DataFrame(paper_rows)

display(paper_table_df)

paper_table_df.to_csv(
    "results/observation_noise/noise_sweep_paper_table.csv",
    index=False,
)


In [ ]:

# ------------------------------------------------------------
# Plot 1: validation margin vs noise
# ------------------------------------------------------------

plt.figure(figsize=(7, 5))

plt.errorbar(
    summary_df["sigma_noise"],
    summary_df["margin_mean"],
    yerr=summary_df["margin_sd"],
    marker="o",
    capsize=4,
)

plt.axhline(0, linewidth=1)
plt.xlabel("Observation-noise standard deviation")
plt.ylabel("Validation margin")
plt.title("Validation margin versus observation noise")
plt.grid(True)
plt.show()


In [ ]:

# ------------------------------------------------------------
# Plot 2: clean test loss vs noise
# ------------------------------------------------------------

plt.figure(figsize=(7, 5))

plt.errorbar(
    summary_df["sigma_noise"],
    summary_df["clean_test_mean"],
    yerr=summary_df["clean_test_sd"],
    marker="o",
    capsize=4,
)

plt.yscale("log")
plt.xlabel("Observation-noise standard deviation")
plt.ylabel("Clean test loss")
plt.title("Recovery of the clean target under noisy observations")
plt.grid(True)
plt.show()



## Interpretation guide

The principal selection quantity is the noisy-validation margin
\[
\widehat\gamma_{\rm noise}
=
\min_{G\neq AB}
L_{\rm val}^{\rm noisy}(G)
-
L_{\rm val}^{\rm noisy}(AB).
\]

- A positive margin means that \(AB\) remains the best candidate even when validation outputs are noisy.
- The clean test loss evaluates the learned predictor against the underlying noiseless target, even though training used noisy labels.
- The noisy test loss measures prediction error relative to the noisy observations themselves.

If the margin decreases and crosses zero as \(\sigma_{\rm noise}\) increases, this gives an empirical breakdown region.

If all tested noise levels retain positive margins, the conclusion should instead be phrased conservatively: selection remained stable up to the largest tested wrapped-Gaussian noise level; no universal noise threshold is claimed.
